# VíaSegura AI

## Notebook 02: Índice de criticidad y zonas críticas de siniestralidad vial

Este notebook toma como base el archivo limpio de siniestros viales de Bogotá para el periodo 2016–2019 y construye una primera identificación de zonas críticas mediante agregación espacial, puntaje de gravedad y mapas interactivos.

La base principal usada es:

data/processed/accidentes_bogota_2016_2019_limpio.csv

In [1]:
import pandas as pd
import geopandas as gpd
import folium
from folium.plugins import HeatMap
import matplotlib.pyplot as plt
from pathlib import Path

print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


In [2]:
import sys
from pathlib import Path

# Localizar config.py (compatible con VS Code, JupyterLab, cualquier cwd)
_root = next(
    (c for c in [Path.cwd(), Path.cwd().parent] if (c / 'config.py').exists()),
    None
)
if _root is None:
    raise RuntimeError(f"No se encontro config.py. cwd={Path.cwd()}")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from config import PROJECT_ROOT, DATA_RAW, DATA_PROCESSED, REPORTS, MAPS

# Alias para mantener compatibilidad con el resto del notebook
ruta_data_processed  = DATA_PROCESSED
ruta_outputs_reports = REPORTS
ruta_outputs_maps    = MAPS
ruta_outputs_charts  = PROJECT_ROOT / "outputs" / "charts"

# Crear carpetas si no existen
ruta_outputs_charts.mkdir(parents=True, exist_ok=True)

print("Raiz del proyecto:", PROJECT_ROOT)
print("Datos procesados: ", ruta_data_processed)
print("Reportes:         ", ruta_outputs_reports)
print("Mapas:            ", ruta_outputs_maps)

Raiz del proyecto: C:\Users\jorge\Documents\viasegura_ai
Datos procesados:  C:\Users\jorge\Documents\viasegura_ai\data\processed
Reportes:          C:\Users\jorge\Documents\viasegura_ai\outputs\reports
Mapas:             C:\Users\jorge\Documents\viasegura_ai\outputs\maps


In [3]:
archivo_limpio = ruta_data_processed / "accidentes_bogota_2016_2019_limpio.csv"

print("Archivo a cargar:")
print(archivo_limpio)
print("Existe:", archivo_limpio.exists())

Archivo a cargar:
C:\Users\jorge\Documents\viasegura_ai\data\processed\accidentes_bogota_2016_2019_limpio.csv
Existe: True


In [4]:
df = pd.read_csv(archivo_limpio)

print("Base cargada correctamente.")
print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])

df.head()

Base cargada correctamente.
Filas: 260831
Columnas: 19


,OBJECTID,FORMULARIO,CODIGO_ACCIDENTE,FECHA_OCURRENCIA_ACC,HORA_OCURRENCIA_ACC,ANO_OCURRENCIA_ACC,MES_OCURRENCIA_ACC,DIA_OCURRENCIA_ACC,DIRECCION,GRAVEDAD,CLASE_ACC,LOCALIDAD,MUNICIPIO,LATITUD,LONGITUD,BARRIO,MVINOMBRE,DISTANCIA_VIA,puntaje_gravedad
0,8,A000402862,4447845,2016-06-08,21:30:00,2016,JUNIO,MIERCOLES,KR 27-CL 9 14,SOLO DANOS,CHOQUE,LOS MARTIRES,BOGOTA DC,4.607648,-74.092901,RICAURTE,AVENIDA GENERAL SANTANDER,0.0,1
1,13,A000551010,4468708,2016-12-27,19:00:00,2016,DICIEMBRE,MARTES,CL 69A-KR 89A 02,CON HERIDOS,CHOQUE,ENGATIVA,BOGOTA DC,4.693578,-74.110585,FLORIDA BLANCA,SIN_NMG,0.2,3
2,17,A22245,4467629,2016-12-16,15:00:00,2016,DICIEMBRE,VIERNES,KR 21-CL 127 03,SOLO DANOS,CHOQUE,USAQUEN,BOGOTA DC,4.706539,-74.051613,LA CALLEJA,SIN_NMG,0.0,1
3,28,A000473024,4459693,2016-10-04,12:15:00,2016,OCTUBRE,MARTES,KR 77-CL 19 02,SOLO DANOS,CHOQUE,FONTIBON,BOGOTA DC,4.654839,-74.128700,CIUDAD HAYUELOS,SIN_NMG,6.7,1
4,32,A000403683,4448989,2016-06-20,11:50:00,2016,JUNIO,LUNES,KR 11-CL 72 02,SOLO DANOS,CHOQUE,CHAPINERO,BOGOTA DC,4.657149,-74.058813,PORCIUNCULA,AVENIDA GERMAN ARCINIEGAS,0.0,1


In [5]:
print("Filas:", len(df))
print("Columnas:", df.shape[1])

print("\nColumnas disponibles:")
for col in df.columns:
    print("-", col)

Filas: 260831
Columnas: 19

Columnas disponibles:
- OBJECTID
- FORMULARIO
- CODIGO_ACCIDENTE
- FECHA_OCURRENCIA_ACC
- HORA_OCURRENCIA_ACC
- ANO_OCURRENCIA_ACC
- MES_OCURRENCIA_ACC
- DIA_OCURRENCIA_ACC
- DIRECCION
- GRAVEDAD
- CLASE_ACC
- LOCALIDAD
- MUNICIPIO
- LATITUD
- LONGITUD
- BARRIO
- MVINOMBRE
- DISTANCIA_VIA
- puntaje_gravedad


In [6]:
print("Duplicados por OBJECTID:", df["OBJECTID"].duplicated().sum())
print("Duplicados por CODIGO_ACCIDENTE:", df["CODIGO_ACCIDENTE"].duplicated().sum())
print("Duplicados por FORMULARIO:", df["FORMULARIO"].duplicated().sum())

Duplicados por OBJECTID: 0
Duplicados por CODIGO_ACCIDENTE: 0
Duplicados por FORMULARIO: 0


In [7]:
print("Nulos en LATITUD:", df["LATITUD"].isnull().sum())
print("Nulos en LONGITUD:", df["LONGITUD"].isnull().sum())
print("Nulos en puntaje_gravedad:", df["puntaje_gravedad"].isnull().sum())

print("\nRango de coordenadas:")
print("Latitud mínima:", df["LATITUD"].min())
print("Latitud máxima:", df["LATITUD"].max())
print("Longitud mínima:", df["LONGITUD"].min())
print("Longitud máxima:", df["LONGITUD"].max())

Nulos en LATITUD: 0
Nulos en LONGITUD: 0
Nulos en puntaje_gravedad: 0

Rango de coordenadas:
Latitud mínima: 4.303100000000001
Latitud máxima: 4.82375217
Longitud mínima: -74.21498272
Longitud máxima: -74.0139


In [8]:
print("Registros por año:")
print(df["ANO_OCURRENCIA_ACC"].value_counts().sort_index())

print("\nRegistros por gravedad:")
print(df["GRAVEDAD"].value_counts())

print("\nPuntajes de gravedad:")
print(df[["GRAVEDAD", "puntaje_gravedad"]].drop_duplicates().sort_values("puntaje_gravedad"))

Registros por año:
ANO_OCURRENCIA_ACC
2016    63932
2017    64828
2018    66816
2019    65255
Name: count, dtype: int64

Registros por gravedad:
GRAVEDAD
SOLO DANOS     171904
CON HERIDOS     84730
CON MUERTOS      4197
Name: count, dtype: int64

Puntajes de gravedad:
        GRAVEDAD  puntaje_gravedad
0     SOLO DANOS                 1
1    CON HERIDOS                 3
153  CON MUERTOS                 5


In [9]:
df_hotspots = df.copy()

# Redondeo a 3 decimales: zonas aproximadas de ~100 m
df_hotspots["LAT_ZONA"] = df_hotspots["LATITUD"].round(3)
df_hotspots["LON_ZONA"] = df_hotspots["LONGITUD"].round(3)

print("Base preparada para análisis espacial.")
print(df_hotspots[["LATITUD", "LONGITUD", "LAT_ZONA", "LON_ZONA"]].head())

Base preparada para análisis espacial.
    LATITUD   LONGITUD  LAT_ZONA  LON_ZONA
0  4.607648 -74.092901     4.608   -74.093
1  4.693578 -74.110585     4.694   -74.111
2  4.706539 -74.051613     4.707   -74.052
3  4.654839 -74.128700     4.655   -74.129
4  4.657149 -74.058813     4.657   -74.059


In [10]:
def valor_mas_frecuente(serie):
    moda = serie.mode()
    
    if len(moda) > 0:
        return moda.iloc[0]
    else:
        return None

In [11]:
zonas_criticas = (
    df_hotspots
    .groupby(["LAT_ZONA", "LON_ZONA"])
    .agg(
        cantidad_siniestros=("OBJECTID", "count"),
        criticidad_total=("puntaje_gravedad", "sum"),
        criticidad_promedio=("puntaje_gravedad", "mean"),
        localidad_predominante=("LOCALIDAD", valor_mas_frecuente),
        barrio_predominante=("BARRIO", valor_mas_frecuente),
        via_predominante=("MVINOMBRE", valor_mas_frecuente),
        clase_predominante=("CLASE_ACC", valor_mas_frecuente),
        gravedad_predominante=("GRAVEDAD", valor_mas_frecuente)
    )
    .reset_index()
    .sort_values("criticidad_total", ascending=False)
)

print("Número de zonas agrupadas:", len(zonas_criticas))

zonas_criticas.head(20)

Número de zonas agrupadas: 17130


,LAT_ZONA,LON_ZONA,cantidad_siniestros,criticidad_total,criticidad_promedio,localidad_predominante,barrio_predominante,via_predominante,clase_predominante,gravedad_predominante
7747,4.632,-74.154,484,736,1.520661,KENNEDY,CIUDAD KENNEDY NORTE,SIN_NMG,CHOQUE,SOLO DANOS
7660,4.631,-74.138,453,693,1.529801,KENNEDY,LAS DOS AVENIDAS,AVENIDA BOYACA,CHOQUE,SOLO DANOS
1697,4.562,-74.139,468,692,1.478632,CIUDAD BOLIVAR,RONDA,AVENIDA BOYACA,CHOQUE,SOLO DANOS
4114,4.597,-74.179,466,686,1.472103,BOSA,CORREDOR FERREO DEL SUR,AVENIDA DEL SUR,CHOQUE,SOLO DANOS
8944,4.645,-74.132,396,644,1.626263,KENNEDY,NUEVO TECHO,AVENIDA BOYACA,CHOQUE,SOLO DANOS
11487,4.679,-74.120,442,574,1.298643,FONTIBON,SANTA CECILIA,AVENIDA CIUDAD DE CALI,CHOQUE,SOLO DANOS
4113,4.597,-74.180,366,554,1.513661,BOSA,CORREDOR FERREO DEL SUR,AVENIDA DEL SUR,CHOQUE,SOLO DANOS
8528,4.640,-74.116,306,542,1.771242,PUENTE ARANDA,GRANJAS DE TECHO,AVENIDA DEL CONGRESO EUCARISTICO,CHOQUE,SOLO DANOS
7341,4.628,-74.171,260,528,2.030769,KENNEDY,LAS MARGARITAS,AVENIDA CIUDAD DE CALI,CHOQUE,SOLO DANOS
6941,4.624,-74.124,312,516,1.653846,PUENTE ARANDA,HIPOTECHO SUR,AVENIDA DEL CONGRESO EUCARISTICO,CHOQUE,SOLO DANOS


In [12]:
def valor_mas_frecuente(serie):
    moda = serie.mode()
    
    if len(moda) > 0:
        return moda.iloc[0]
    else:
        return None

In [13]:
zonas_criticas = (
    df_hotspots
    .groupby(["LAT_ZONA", "LON_ZONA"])
    .agg(
        cantidad_siniestros=("OBJECTID", "count"),
        criticidad_total=("puntaje_gravedad", "sum"),
        criticidad_promedio=("puntaje_gravedad", "mean"),
        localidad_predominante=("LOCALIDAD", valor_mas_frecuente),
        barrio_predominante=("BARRIO", valor_mas_frecuente),
        via_predominante=("MVINOMBRE", valor_mas_frecuente),
        clase_predominante=("CLASE_ACC", valor_mas_frecuente),
        gravedad_predominante=("GRAVEDAD", valor_mas_frecuente)
    )
    .reset_index()
    .sort_values("criticidad_total", ascending=False)
)

print("Número de zonas agrupadas:", len(zonas_criticas))

zonas_criticas.head(20)

Número de zonas agrupadas: 17130


,LAT_ZONA,LON_ZONA,cantidad_siniestros,criticidad_total,criticidad_promedio,localidad_predominante,barrio_predominante,via_predominante,clase_predominante,gravedad_predominante
7747,4.632,-74.154,484,736,1.520661,KENNEDY,CIUDAD KENNEDY NORTE,SIN_NMG,CHOQUE,SOLO DANOS
7660,4.631,-74.138,453,693,1.529801,KENNEDY,LAS DOS AVENIDAS,AVENIDA BOYACA,CHOQUE,SOLO DANOS
1697,4.562,-74.139,468,692,1.478632,CIUDAD BOLIVAR,RONDA,AVENIDA BOYACA,CHOQUE,SOLO DANOS
4114,4.597,-74.179,466,686,1.472103,BOSA,CORREDOR FERREO DEL SUR,AVENIDA DEL SUR,CHOQUE,SOLO DANOS
8944,4.645,-74.132,396,644,1.626263,KENNEDY,NUEVO TECHO,AVENIDA BOYACA,CHOQUE,SOLO DANOS
11487,4.679,-74.120,442,574,1.298643,FONTIBON,SANTA CECILIA,AVENIDA CIUDAD DE CALI,CHOQUE,SOLO DANOS
4113,4.597,-74.180,366,554,1.513661,BOSA,CORREDOR FERREO DEL SUR,AVENIDA DEL SUR,CHOQUE,SOLO DANOS
8528,4.640,-74.116,306,542,1.771242,PUENTE ARANDA,GRANJAS DE TECHO,AVENIDA DEL CONGRESO EUCARISTICO,CHOQUE,SOLO DANOS
7341,4.628,-74.171,260,528,2.030769,KENNEDY,LAS MARGARITAS,AVENIDA CIUDAD DE CALI,CHOQUE,SOLO DANOS
6941,4.624,-74.124,312,516,1.653846,PUENTE ARANDA,HIPOTECHO SUR,AVENIDA DEL CONGRESO EUCARISTICO,CHOQUE,SOLO DANOS


In [14]:
top20_zonas_criticas = zonas_criticas.head(20)

top20_zonas_criticas

,LAT_ZONA,LON_ZONA,cantidad_siniestros,criticidad_total,criticidad_promedio,localidad_predominante,barrio_predominante,via_predominante,clase_predominante,gravedad_predominante
7747,4.632,-74.154,484,736,1.520661,KENNEDY,CIUDAD KENNEDY NORTE,SIN_NMG,CHOQUE,SOLO DANOS
7660,4.631,-74.138,453,693,1.529801,KENNEDY,LAS DOS AVENIDAS,AVENIDA BOYACA,CHOQUE,SOLO DANOS
1697,4.562,-74.139,468,692,1.478632,CIUDAD BOLIVAR,RONDA,AVENIDA BOYACA,CHOQUE,SOLO DANOS
4114,4.597,-74.179,466,686,1.472103,BOSA,CORREDOR FERREO DEL SUR,AVENIDA DEL SUR,CHOQUE,SOLO DANOS
8944,4.645,-74.132,396,644,1.626263,KENNEDY,NUEVO TECHO,AVENIDA BOYACA,CHOQUE,SOLO DANOS
11487,4.679,-74.120,442,574,1.298643,FONTIBON,SANTA CECILIA,AVENIDA CIUDAD DE CALI,CHOQUE,SOLO DANOS
4113,4.597,-74.180,366,554,1.513661,BOSA,CORREDOR FERREO DEL SUR,AVENIDA DEL SUR,CHOQUE,SOLO DANOS
8528,4.640,-74.116,306,542,1.771242,PUENTE ARANDA,GRANJAS DE TECHO,AVENIDA DEL CONGRESO EUCARISTICO,CHOQUE,SOLO DANOS
7341,4.628,-74.171,260,528,2.030769,KENNEDY,LAS MARGARITAS,AVENIDA CIUDAD DE CALI,CHOQUE,SOLO DANOS
6941,4.624,-74.124,312,516,1.653846,PUENTE ARANDA,HIPOTECHO SUR,AVENIDA DEL CONGRESO EUCARISTICO,CHOQUE,SOLO DANOS


In [15]:
zonas_criticas.to_csv(
    ruta_outputs_reports / "zonas_criticas_siniestros_2016_2019.csv",
    index=False
)

top20_zonas_criticas.to_csv(
    ruta_outputs_reports / "top20_zonas_criticas_siniestros_2016_2019.csv",
    index=False
)

print("Tablas de zonas críticas guardadas correctamente.")

Tablas de zonas críticas guardadas correctamente.


In [16]:
top50_zonas = zonas_criticas.head(50).copy()

mapa_top50 = folium.Map(
    location=[4.65, -74.08],
    zoom_start=11,
    tiles="CartoDB positron"
)

max_criticidad = top50_zonas["criticidad_total"].max()

for _, row in top50_zonas.iterrows():
    
    radio = 5 + (row["criticidad_total"] / max_criticidad) * 20
    
    popup_texto = f"""
    <b>Zona crítica aproximada</b><br>
    <b>Localidad:</b> {row['localidad_predominante']}<br>
    <b>Barrio:</b> {row['barrio_predominante']}<br>
    <b>Vía predominante:</b> {row['via_predominante']}<br>
    <b>Cantidad de siniestros:</b> {row['cantidad_siniestros']}<br>
    <b>Criticidad total:</b> {row['criticidad_total']}<br>
    <b>Criticidad promedio:</b> {round(row['criticidad_promedio'], 2)}<br>
    <b>Clase predominante:</b> {row['clase_predominante']}<br>
    <b>Gravedad predominante:</b> {row['gravedad_predominante']}<br>
    """
    
    folium.CircleMarker(
        location=[row["LAT_ZONA"], row["LON_ZONA"]],
        radius=radio,
        popup=folium.Popup(popup_texto, max_width=350),
        fill=True,
        fill_opacity=0.7
    ).add_to(mapa_top50)

mapa_top50

In [17]:
mapa_top50.save(
    ruta_outputs_maps / "mapa_top50_zonas_criticas_siniestros_2016_2019.html"
)

print("Mapa top 50 guardado correctamente.")

Mapa top 50 guardado correctamente.


In [18]:
columnas_top = [
    "LAT_ZONA",
    "LON_ZONA",
    "cantidad_siniestros",
    "criticidad_total",
    "criticidad_promedio",
    "localidad_predominante",
    "barrio_predominante",
    "via_predominante",
    "clase_predominante",
    "gravedad_predominante"
]

top20_zonas_criticas[columnas_top]

,LAT_ZONA,LON_ZONA,cantidad_siniestros,criticidad_total,criticidad_promedio,localidad_predominante,barrio_predominante,via_predominante,clase_predominante,gravedad_predominante
7747,4.632,-74.154,484,736,1.520661,KENNEDY,CIUDAD KENNEDY NORTE,SIN_NMG,CHOQUE,SOLO DANOS
7660,4.631,-74.138,453,693,1.529801,KENNEDY,LAS DOS AVENIDAS,AVENIDA BOYACA,CHOQUE,SOLO DANOS
1697,4.562,-74.139,468,692,1.478632,CIUDAD BOLIVAR,RONDA,AVENIDA BOYACA,CHOQUE,SOLO DANOS
4114,4.597,-74.179,466,686,1.472103,BOSA,CORREDOR FERREO DEL SUR,AVENIDA DEL SUR,CHOQUE,SOLO DANOS
8944,4.645,-74.132,396,644,1.626263,KENNEDY,NUEVO TECHO,AVENIDA BOYACA,CHOQUE,SOLO DANOS
11487,4.679,-74.120,442,574,1.298643,FONTIBON,SANTA CECILIA,AVENIDA CIUDAD DE CALI,CHOQUE,SOLO DANOS
4113,4.597,-74.180,366,554,1.513661,BOSA,CORREDOR FERREO DEL SUR,AVENIDA DEL SUR,CHOQUE,SOLO DANOS
8528,4.640,-74.116,306,542,1.771242,PUENTE ARANDA,GRANJAS DE TECHO,AVENIDA DEL CONGRESO EUCARISTICO,CHOQUE,SOLO DANOS
7341,4.628,-74.171,260,528,2.030769,KENNEDY,LAS MARGARITAS,AVENIDA CIUDAD DE CALI,CHOQUE,SOLO DANOS
6941,4.624,-74.124,312,516,1.653846,PUENTE ARANDA,HIPOTECHO SUR,AVENIDA DEL CONGRESO EUCARISTICO,CHOQUE,SOLO DANOS


In [19]:
zonas_criticas.to_csv(
    ruta_outputs_reports / "zonas_criticas_siniestros_2016_2019.csv",
    index=False
)

top20_zonas_criticas.to_csv(
    ruta_outputs_reports / "top20_zonas_criticas_siniestros_2016_2019.csv",
    index=False
)

print("Tablas de zonas críticas guardadas correctamente.")

Tablas de zonas críticas guardadas correctamente.


In [20]:
top50_zonas = zonas_criticas.head(50).copy()

mapa_top50 = folium.Map(
    location=[4.65, -74.08],
    zoom_start=11,
    tiles="CartoDB positron"
)

max_criticidad = top50_zonas["criticidad_total"].max()

for _, row in top50_zonas.iterrows():
    
    radio = 5 + (row["criticidad_total"] / max_criticidad) * 20
    
    popup_texto = f"""
    <b>Zona crítica aproximada</b><br><br>
    <b>Localidad:</b> {row['localidad_predominante']}<br>
    <b>Barrio:</b> {row['barrio_predominante']}<br>
    <b>Vía predominante:</b> {row['via_predominante']}<br><br>
    <b>Cantidad de siniestros:</b> {row['cantidad_siniestros']}<br>
    <b>Criticidad total:</b> {row['criticidad_total']}<br>
    <b>Criticidad promedio:</b> {round(row['criticidad_promedio'], 2)}<br><br>
    <b>Clase predominante:</b> {row['clase_predominante']}<br>
    <b>Gravedad predominante:</b> {row['gravedad_predominante']}<br><br>
    <b>Coordenada aproximada:</b> {row['LAT_ZONA']}, {row['LON_ZONA']}
    """
    
    folium.CircleMarker(
        location=[row["LAT_ZONA"], row["LON_ZONA"]],
        radius=radio,
        popup=folium.Popup(popup_texto, max_width=350),
        fill=True,
        fill_opacity=0.7
    ).add_to(mapa_top50)

mapa_top50

In [21]:
mapa_top50.save(
    ruta_outputs_maps / "mapa_top50_zonas_criticas_siniestros_2016_2019.html"
)

print("Mapa top 50 zonas críticas guardado correctamente.")

Mapa top 50 zonas críticas guardado correctamente.


## Nota metodológica sobre zonas críticas

Las zonas críticas se construyeron mediante agregación espacial de coordenadas redondeadas a tres decimales. Esta aproximación permite agrupar siniestros ocurridos en ubicaciones cercanas, aproximadamente dentro de áreas del orden de 100 metros.

El indicador de criticidad total corresponde a la suma del puntaje de gravedad asignado a los siniestros registrados en cada zona:

- Solo daños: 1 punto
- Con heridos: 3 puntos
- Con muertos: 5 puntos

Los resultados deben interpretarse como una priorización exploratoria de concentración y severidad de siniestros registrados, no como una medición definitiva de riesgo vial. Para estimar riesgo real sería necesario incorporar variables de exposición, como flujo vehicular, población, longitud de red vial, velocidad, diseño geométrico, control semafórico y condiciones del entorno urbano.

Además, el agrupamiento por coordenadas redondeadas puede dividir corredores o intersecciones cercanas en varias zonas contiguas. Por esta razón, las zonas identificadas deben entenderse como áreas aproximadas de concentración, no como intersecciones oficiales.

In [22]:
# Crear tabla por zona y año
zona_anio = (
    df_hotspots
    .groupby(["LAT_ZONA", "LON_ZONA", "ANO_OCURRENCIA_ACC"])
    .agg(
        cantidad_siniestros=("OBJECTID", "count"),
        criticidad_total=("puntaje_gravedad", "sum"),
        criticidad_promedio=("puntaje_gravedad", "mean")
    )
    .reset_index()
)

zona_anio.head()

,LAT_ZONA,LON_ZONA,ANO_OCURRENCIA_ACC,cantidad_siniestros,criticidad_total,criticidad_promedio
0,4.303,-74.034,2016,2,10,5.0
1,4.460,-74.118,2019,2,6,3.0
2,4.464,-74.123,2017,2,2,1.0
3,4.465,-74.129,2018,12,24,2.0
4,4.465,-74.129,2019,4,8,2.0


In [23]:
# Matriz de criticidad por año
criticidad_por_anio = zona_anio.pivot_table(
    index=["LAT_ZONA", "LON_ZONA"],
    columns="ANO_OCURRENCIA_ACC",
    values="criticidad_total",
    fill_value=0
).reset_index()

criticidad_por_anio.columns.name = None

criticidad_por_anio.head()

,LAT_ZONA,LON_ZONA,2016,2017,2018,2019
0,4.303,-74.034,10.0,0.0,0.0,0.0
1,4.460,-74.118,0.0,0.0,0.0,6.0
2,4.464,-74.123,0.0,2.0,0.0,0.0
3,4.465,-74.129,0.0,0.0,24.0,8.0
4,4.465,-74.123,6.0,2.0,0.0,0.0


In [24]:
criticidad_por_anio = criticidad_por_anio.rename(columns={
    2016: "criticidad_2016",
    2017: "criticidad_2017",
    2018: "criticidad_2018",
    2019: "criticidad_2019"
})

criticidad_por_anio.head()

,LAT_ZONA,LON_ZONA,criticidad_2016,criticidad_2017,criticidad_2018,criticidad_2019
0,4.303,-74.034,10.0,0.0,0.0,0.0
1,4.460,-74.118,0.0,0.0,0.0,6.0
2,4.464,-74.123,0.0,2.0,0.0,0.0
3,4.465,-74.129,0.0,0.0,24.0,8.0
4,4.465,-74.123,6.0,2.0,0.0,0.0


In [25]:
columnas_anios = [
    "criticidad_2016",
    "criticidad_2017",
    "criticidad_2018",
    "criticidad_2019"
]

criticidad_por_anio["anios_activos"] = (
    criticidad_por_anio[columnas_anios] > 0
).sum(axis=1)

criticidad_por_anio.head()

,LAT_ZONA,LON_ZONA,criticidad_2016,criticidad_2017,criticidad_2018,criticidad_2019,anios_activos
0,4.303,-74.034,10.0,0.0,0.0,0.0,1
1,4.460,-74.118,0.0,0.0,0.0,6.0,1
2,4.464,-74.123,0.0,2.0,0.0,0.0,1
3,4.465,-74.129,0.0,0.0,24.0,8.0,2
4,4.465,-74.123,6.0,2.0,0.0,0.0,2


In [26]:
zonas_diagnostico = zonas_criticas.merge(
    criticidad_por_anio,
    on=["LAT_ZONA", "LON_ZONA"],
    how="left"
)

zonas_diagnostico.head()

,LAT_ZONA,LON_ZONA,cantidad_siniestros,criticidad_total,criticidad_promedio,localidad_predominante,barrio_predominante,via_predominante,clase_predominante,gravedad_predominante,criticidad_2016,criticidad_2017,criticidad_2018,criticidad_2019,anios_activos
0,4.632,-74.154,484,736,1.520661,KENNEDY,CIUDAD KENNEDY NORTE,SIN_NMG,CHOQUE,SOLO DANOS,180.0,198.0,198.0,160.0,4
1,4.631,-74.138,453,693,1.529801,KENNEDY,LAS DOS AVENIDAS,AVENIDA BOYACA,CHOQUE,SOLO DANOS,92.0,125.0,242.0,234.0,4
2,4.562,-74.139,468,692,1.478632,CIUDAD BOLIVAR,RONDA,AVENIDA BOYACA,CHOQUE,SOLO DANOS,214.0,186.0,154.0,138.0,4
3,4.597,-74.179,466,686,1.472103,BOSA,CORREDOR FERREO DEL SUR,AVENIDA DEL SUR,CHOQUE,SOLO DANOS,124.0,232.0,180.0,150.0,4
4,4.645,-74.132,396,644,1.626263,KENNEDY,NUEVO TECHO,AVENIDA BOYACA,CHOQUE,SOLO DANOS,122.0,180.0,162.0,180.0,4


In [27]:
def clasificar_hotspot(row):
    cantidad = row["cantidad_siniestros"]
    criticidad_promedio = row["criticidad_promedio"]
    anios_activos = row["anios_activos"]
    
    if anios_activos >= 4 and cantidad >= 300:
        return "Hotspot estructural persistente"
    
    elif criticidad_promedio >= 2.0 and cantidad >= 150:
        return "Hotspot severo"
    
    elif cantidad >= 350 and criticidad_promedio < 1.7:
        return "Hotspot de alto volumen"
    
    elif anios_activos >= 3 and criticidad_promedio >= 1.7:
        return "Hotspot persistente con severidad media-alta"
    
    else:
        return "Hotspot exploratorio"

zonas_diagnostico["tipo_hotspot"] = zonas_diagnostico.apply(clasificar_hotspot, axis=1)

zonas_diagnostico[[
    "LAT_ZONA",
    "LON_ZONA",
    "cantidad_siniestros",
    "criticidad_total",
    "criticidad_promedio",
    "anios_activos",
    "tipo_hotspot",
    "localidad_predominante",
    "via_predominante"
]].head(20)

,LAT_ZONA,LON_ZONA,cantidad_siniestros,criticidad_total,criticidad_promedio,anios_activos,tipo_hotspot,localidad_predominante,via_predominante
0,4.632,-74.154,484,736,1.520661,4,Hotspot estructural persistente,KENNEDY,SIN_NMG
1,4.631,-74.138,453,693,1.529801,4,Hotspot estructural persistente,KENNEDY,AVENIDA BOYACA
2,4.562,-74.139,468,692,1.478632,4,Hotspot estructural persistente,CIUDAD BOLIVAR,AVENIDA BOYACA
3,4.597,-74.179,466,686,1.472103,4,Hotspot estructural persistente,BOSA,AVENIDA DEL SUR
4,4.645,-74.132,396,644,1.626263,4,Hotspot estructural persistente,KENNEDY,AVENIDA BOYACA
5,4.679,-74.120,442,574,1.298643,4,Hotspot estructural persistente,FONTIBON,AVENIDA CIUDAD DE CALI
6,4.597,-74.180,366,554,1.513661,4,Hotspot estructural persistente,BOSA,AVENIDA DEL SUR
7,4.640,-74.116,306,542,1.771242,4,Hotspot estructural persistente,PUENTE ARANDA,AVENIDA DEL CONGRESO EUCARISTICO
8,4.628,-74.171,260,528,2.030769,4,Hotspot severo,KENNEDY,AVENIDA CIUDAD DE CALI
9,4.624,-74.124,312,516,1.653846,4,Hotspot estructural persistente,PUENTE ARANDA,AVENIDA DEL CONGRESO EUCARISTICO


In [28]:
def asignar_prioridad(row):
    tipo = row["tipo_hotspot"]
    criticidad_total = row["criticidad_total"]
    anios_activos = row["anios_activos"]
    
    if tipo == "Hotspot estructural persistente" and criticidad_total >= 500:
        return "Prioridad 1 - Intervención prioritaria"
    
    elif tipo == "Hotspot severo":
        return "Prioridad 1 - Revisión urgente por severidad"
    
    elif tipo == "Hotspot persistente con severidad media-alta":
        return "Prioridad 2 - Auditoría de seguridad vial"
    
    elif tipo == "Hotspot de alto volumen":
        return "Prioridad 2 - Gestión operativa y control"
    
    else:
        return "Prioridad 3 - Seguimiento"

zonas_diagnostico["prioridad_intervencion"] = zonas_diagnostico.apply(asignar_prioridad, axis=1)

zonas_diagnostico[[
    "LAT_ZONA",
    "LON_ZONA",
    "cantidad_siniestros",
    "criticidad_total",
    "criticidad_promedio",
    "anios_activos",
    "tipo_hotspot",
    "prioridad_intervencion",
    "localidad_predominante",
    "barrio_predominante",
    "via_predominante"
]].head(20)

,LAT_ZONA,LON_ZONA,cantidad_siniestros,criticidad_total,criticidad_promedio,anios_activos,tipo_hotspot,prioridad_intervencion,localidad_predominante,barrio_predominante,via_predominante
0,4.632,-74.154,484,736,1.520661,4,Hotspot estructural persistente,Prioridad 1 - Intervención prioritaria,KENNEDY,CIUDAD KENNEDY NORTE,SIN_NMG
1,4.631,-74.138,453,693,1.529801,4,Hotspot estructural persistente,Prioridad 1 - Intervención prioritaria,KENNEDY,LAS DOS AVENIDAS,AVENIDA BOYACA
2,4.562,-74.139,468,692,1.478632,4,Hotspot estructural persistente,Prioridad 1 - Intervención prioritaria,CIUDAD BOLIVAR,RONDA,AVENIDA BOYACA
3,4.597,-74.179,466,686,1.472103,4,Hotspot estructural persistente,Prioridad 1 - Intervención prioritaria,BOSA,CORREDOR FERREO DEL SUR,AVENIDA DEL SUR
4,4.645,-74.132,396,644,1.626263,4,Hotspot estructural persistente,Prioridad 1 - Intervención prioritaria,KENNEDY,NUEVO TECHO,AVENIDA BOYACA
5,4.679,-74.120,442,574,1.298643,4,Hotspot estructural persistente,Prioridad 1 - Intervención prioritaria,FONTIBON,SANTA CECILIA,AVENIDA CIUDAD DE CALI
6,4.597,-74.180,366,554,1.513661,4,Hotspot estructural persistente,Prioridad 1 - Intervención prioritaria,BOSA,CORREDOR FERREO DEL SUR,AVENIDA DEL SUR
7,4.640,-74.116,306,542,1.771242,4,Hotspot estructural persistente,Prioridad 1 - Intervención prioritaria,PUENTE ARANDA,GRANJAS DE TECHO,AVENIDA DEL CONGRESO EUCARISTICO
8,4.628,-74.171,260,528,2.030769,4,Hotspot severo,Prioridad 1 - Revisión urgente por severidad,KENNEDY,LAS MARGARITAS,AVENIDA CIUDAD DE CALI
9,4.624,-74.124,312,516,1.653846,4,Hotspot estructural persistente,Prioridad 1 - Intervención prioritaria,PUENTE ARANDA,HIPOTECHO SUR,AVENIDA DEL CONGRESO EUCARISTICO


In [29]:
resumen_tipo_hotspot = (
    zonas_diagnostico
    .groupby("tipo_hotspot")
    .agg(
        cantidad_zonas=("tipo_hotspot", "count"),
        criticidad_total_acumulada=("criticidad_total", "sum"),
        siniestros_acumulados=("cantidad_siniestros", "sum")
    )
    .reset_index()
    .sort_values("criticidad_total_acumulada", ascending=False)
)

resumen_tipo_hotspot

,tipo_hotspot,cantidad_zonas,criticidad_total_acumulada,siniestros_acumulados
1,Hotspot exploratorio,13137,233369,155003
2,Hotspot persistente con severidad media-alta,3965,199753,96823
0,Hotspot estructural persistente,21,11195,7695
3,Hotspot severo,7,2762,1310


In [30]:
resumen_prioridad = (
    zonas_diagnostico
    .groupby("prioridad_intervencion")
    .agg(
        cantidad_zonas=("prioridad_intervencion", "count"),
        criticidad_total_acumulada=("criticidad_total", "sum"),
        siniestros_acumulados=("cantidad_siniestros", "sum")
    )
    .reset_index()
    .sort_values("criticidad_total_acumulada", ascending=False)
)

resumen_prioridad

,prioridad_intervencion,cantidad_zonas,criticidad_total_acumulada,siniestros_acumulados
3,Prioridad 3 - Seguimiento,13149,238927,159005
2,Prioridad 2 - Auditoría de seguridad vial,3965,199753,96823
0,Prioridad 1 - Intervención prioritaria,9,5637,3693
1,Prioridad 1 - Revisión urgente por severidad,7,2762,1310


In [31]:
zonas_diagnostico.to_csv(
    ruta_outputs_reports / "zonas_criticas_diagnostico_2016_2019.csv",
    index=False
)

resumen_tipo_hotspot.to_csv(
    ruta_outputs_reports / "resumen_tipo_hotspot_2016_2019.csv",
    index=False
)

resumen_prioridad.to_csv(
    ruta_outputs_reports / "resumen_prioridad_intervencion_2016_2019.csv",
    index=False
)

print("Tablas diagnósticas guardadas correctamente.")

Tablas diagnósticas guardadas correctamente.


In [32]:
# Conteo de tipos de gravedad por zona
conteo_gravedad_zona = (
    df_hotspots
    .groupby(["LAT_ZONA", "LON_ZONA", "GRAVEDAD"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

# Asegurar que existan las columnas aunque alguna categoría no aparezca
for col in ["SOLO DANOS", "SOLO DAÑOS", "CON HERIDOS", "CON MUERTOS"]:
    if col not in conteo_gravedad_zona.columns:
        conteo_gravedad_zona[col] = 0

# Unificar daños por si aparece con o sin Ñ
conteo_gravedad_zona["siniestros_solo_danos"] = (
    conteo_gravedad_zona["SOLO DANOS"] + conteo_gravedad_zona["SOLO DAÑOS"]
)

conteo_gravedad_zona["siniestros_con_heridos"] = conteo_gravedad_zona["CON HERIDOS"]
conteo_gravedad_zona["siniestros_con_muertos"] = conteo_gravedad_zona["CON MUERTOS"]

conteo_gravedad_zona = conteo_gravedad_zona[
    [
        "LAT_ZONA",
        "LON_ZONA",
        "siniestros_solo_danos",
        "siniestros_con_heridos",
        "siniestros_con_muertos"
    ]
]

conteo_gravedad_zona.head()

GRAVEDAD,LAT_ZONA,LON_ZONA,siniestros_solo_danos,siniestros_con_heridos,siniestros_con_muertos
0,4.303,-74.034,0,0,2
1,4.460,-74.118,0,2,0
2,4.464,-74.123,2,0,0
3,4.465,-74.129,8,8,0
4,4.465,-74.123,2,2,0


In [33]:
zonas_avanzado = zonas_diagnostico.merge(
    conteo_gravedad_zona,
    on=["LAT_ZONA", "LON_ZONA"],
    how="left"
)

zonas_avanzado[[
    "LAT_ZONA",
    "LON_ZONA",
    "cantidad_siniestros",
    "criticidad_total",
    "siniestros_solo_danos",
    "siniestros_con_heridos",
    "siniestros_con_muertos"
]].head(20)

,LAT_ZONA,LON_ZONA,cantidad_siniestros,criticidad_total,siniestros_solo_danos,siniestros_con_heridos,siniestros_con_muertos
0,4.632,-74.154,484,736,360,122,2
1,4.631,-74.138,453,693,338,110,5
2,4.562,-74.139,468,692,370,84,14
3,4.597,-74.179,466,686,364,94,8
4,4.645,-74.132,396,644,272,124,0
5,4.679,-74.120,442,574,378,62,2
6,4.597,-74.180,366,554,276,86,4
7,4.640,-74.116,306,542,192,110,4
8,4.628,-74.171,260,528,130,126,4
9,4.624,-74.124,312,516,212,98,2


In [34]:
zonas_avanzado["criticidad_fatalidad_alta"] = (
    zonas_avanzado["siniestros_solo_danos"] * 1 +
    zonas_avanzado["siniestros_con_heridos"] * 5 +
    zonas_avanzado["siniestros_con_muertos"] * 25
)

zonas_avanzado[[
    "LAT_ZONA",
    "LON_ZONA",
    "criticidad_total",
    "criticidad_fatalidad_alta",
    "siniestros_con_muertos"
]].head(20)

,LAT_ZONA,LON_ZONA,criticidad_total,criticidad_fatalidad_alta,siniestros_con_muertos
0,4.632,-74.154,736,1020,2
1,4.631,-74.138,693,1013,5
2,4.562,-74.139,692,1140,14
3,4.597,-74.179,686,1034,8
4,4.645,-74.132,644,892,0
5,4.679,-74.120,574,738,2
6,4.597,-74.180,554,806,4
7,4.640,-74.116,542,842,4
8,4.628,-74.171,528,860,4
9,4.624,-74.124,516,752,2


In [35]:
zonas_avanzado["rank_base"] = zonas_avanzado["criticidad_total"].rank(
    ascending=False,
    method="min"
)

zonas_avanzado["rank_fatalidad_alta"] = zonas_avanzado["criticidad_fatalidad_alta"].rank(
    ascending=False,
    method="min"
)

# Si variacion_ranking es positiva, la zona sube cuando se da más peso a fatalidad
zonas_avanzado["variacion_ranking_fatalidad"] = (
    zonas_avanzado["rank_base"] - zonas_avanzado["rank_fatalidad_alta"]
)

zonas_avanzado[[
    "LAT_ZONA",
    "LON_ZONA",
    "criticidad_total",
    "criticidad_fatalidad_alta",
    "rank_base",
    "rank_fatalidad_alta",
    "variacion_ranking_fatalidad",
    "siniestros_con_muertos",
    "localidad_predominante",
    "via_predominante"
]].sort_values("variacion_ranking_fatalidad", ascending=False).head(20)

,LAT_ZONA,LON_ZONA,criticidad_total,criticidad_fatalidad_alta,rank_base,rank_fatalidad_alta,variacion_ranking_fatalidad,siniestros_con_muertos,localidad_predominante,via_predominante
13253,4.749,-74.125,5,25,13254.0,6617.0,6637.0,1,SUBA,SIN_NMG
13257,4.679,-74.155,5,25,13254.0,6617.0,6637.0,1,FONTIBON,AVENIDA CENTENARIO
13255,4.750,-74.127,5,25,13254.0,6617.0,6637.0,1,SUBA,NaN
13254,4.550,-74.133,5,25,13254.0,6617.0,6637.0,1,CIUDAD BOLIVAR,NaN
9553,4.520,-74.090,10,50,8938.0,3873.0,5065.0,2,SAN CRISTOBAL,SIN_NMG
9615,4.746,-74.055,10,50,8938.0,3873.0,5065.0,2,SUBA,SIN_NMG
9635,4.559,-74.134,10,50,8938.0,3873.0,5065.0,2,CIUDAD BOLIVAR,AVENIDA TUNJUELITO
9626,4.526,-74.141,10,50,8938.0,3873.0,5065.0,2,CIUDAD BOLIVAR,SIN_NMG
9646,4.580,-74.121,10,50,8938.0,3873.0,5065.0,2,RAFAEL URIBE URIBE,SIN_NMG
9619,4.537,-74.151,10,50,8938.0,3873.0,5065.0,2,CIUDAD BOLIVAR,SIN_NMG


In [36]:
zonas_avanzado["score_volumen"] = zonas_avanzado["cantidad_siniestros"].rank(pct=True)
zonas_avanzado["score_criticidad_total"] = zonas_avanzado["criticidad_total"].rank(pct=True)
zonas_avanzado["score_severidad_promedio"] = zonas_avanzado["criticidad_promedio"].rank(pct=True)
zonas_avanzado["score_persistencia"] = zonas_avanzado["anios_activos"] / 4
zonas_avanzado["score_fatalidad"] = zonas_avanzado["siniestros_con_muertos"].rank(pct=True)

scores_cols = ["score_volumen", "score_criticidad_total", "score_severidad_promedio",
               "score_persistencia", "score_fatalidad"]
zonas_avanzado["IPI"] = zonas_avanzado[scores_cols].mean(axis=1) * 100

zonas_avanzado["rank_IPI"] = zonas_avanzado["IPI"].rank(
    ascending=False,
    method="first"
).astype(int)

zonas_avanzado = zonas_avanzado.sort_values("rank_IPI")

zonas_avanzado[[
    "rank_IPI",
    "LAT_ZONA",
    "LON_ZONA",
    "IPI",
    "cantidad_siniestros",
    "criticidad_total",
    "criticidad_promedio",
    "anios_activos",
    "siniestros_con_muertos",
    "tipo_hotspot",
    "localidad_predominante",
    "via_predominante"
]].head(20)


,rank_IPI,LAT_ZONA,LON_ZONA,IPI,cantidad_siniestros,criticidad_total,criticidad_promedio,anios_activos,siniestros_con_muertos,tipo_hotspot,localidad_predominante,via_predominante
83,1,4.576,-74.155,96.275540,108,296,2.740741,4,18,Hotspot persistente con severidad media-alta,CIUDAD BOLIVAR,AVENIDA CIUDAD DE VILLAVICENCIO
135,2,4.602,-74.081,95.778167,96,244,2.541667,4,10,Hotspot persistente con severidad media-alta,SANTA FE,AVENIDA CARACAS
140,3,4.602,-74.077,95.689434,94,238,2.531915,4,6,Hotspot persistente con severidad media-alta,SANTA FE,AVENIDA FERNANDO MAZUERA
197,4,4.605,-74.075,95.084647,86,206,2.395349,4,6,Hotspot persistente con severidad media-alta,SANTA FE,AVENIDA FERNANDO MAZUERA
1224,5,4.642,-74.191,94.994162,24,80,3.333333,4,4,Hotspot persistente con severidad media-alta,BOSA,SIN_NMG
369,6,4.607,-74.130,94.965558,60,152,2.533333,4,10,Hotspot persistente con severidad media-alta,PUENTE ARANDA,AVENIDA DEL CONGRESO EUCARISTICO
27,7,4.611,-74.075,94.701109,182,422,2.318681,4,4,Hotspot severo,SANTA FE,AVENIDA CARACAS
147,8,4.576,-74.120,94.636311,92,236,2.565217,4,2,Hotspot persistente con severidad media-alta,RAFAEL URIBE URIBE,AVENIDA CARACAS
515,9,4.618,-74.170,94.575015,48,132,2.750000,4,4,Hotspot persistente con severidad media-alta,KENNEDY,AVENIDA AGOBERTO MEJIA CIFUENTES
343,10,4.512,-74.115,94.455342,68,160,2.352941,4,4,Hotspot persistente con severidad media-alta,USME,AVENIDA CARACAS


In [37]:
def asignar_prioridad_ipi(rank):
    if rank <= 50:
        return "Prioridad 1 - Intervención prioritaria"
    elif rank <= 200:
        return "Prioridad 2 - Auditoría de seguridad vial"
    elif rank <= 500:
        return "Prioridad 3 - Monitoreo y gestión preventiva"
    else:
        return "Seguimiento"

zonas_avanzado["prioridad_IPI"] = zonas_avanzado["rank_IPI"].apply(asignar_prioridad_ipi)

zonas_avanzado[[
    "rank_IPI",
    "IPI",
    "LAT_ZONA",
    "LON_ZONA",
    "prioridad_IPI",
    "tipo_hotspot",
    "cantidad_siniestros",
    "criticidad_total",
    "criticidad_promedio",
    "anios_activos",
    "siniestros_con_muertos",
    "localidad_predominante",
    "barrio_predominante",
    "via_predominante"
]].head(50)

,rank_IPI,IPI,LAT_ZONA,LON_ZONA,prioridad_IPI,tipo_hotspot,cantidad_siniestros,criticidad_total,criticidad_promedio,anios_activos,siniestros_con_muertos,localidad_predominante,barrio_predominante,via_predominante
83,1,96.275540,4.576,-74.155,Prioridad 1 - Intervención prioritaria,Hotspot persistente con severidad media-alta,108,296,2.740741,4,18,CIUDAD BOLIVAR,VERONA,AVENIDA CIUDAD DE VILLAVICENCIO
135,2,95.778167,4.602,-74.081,Prioridad 1 - Intervención prioritaria,Hotspot persistente con severidad media-alta,96,244,2.541667,4,10,SANTA FE,VOTO NACIONAL,AVENIDA CARACAS
140,3,95.689434,4.602,-74.077,Prioridad 1 - Intervención prioritaria,Hotspot persistente con severidad media-alta,94,238,2.531915,4,6,SANTA FE,LA CAPUCHINA,AVENIDA FERNANDO MAZUERA
197,4,95.084647,4.605,-74.075,Prioridad 1 - Intervención prioritaria,Hotspot persistente con severidad media-alta,86,206,2.395349,4,6,SANTA FE,VERACRUZ,AVENIDA FERNANDO MAZUERA
1224,5,94.994162,4.642,-74.191,Prioridad 1 - Intervención prioritaria,Hotspot persistente con severidad media-alta,24,80,3.333333,4,4,BOSA,EL CORZO I,SIN_NMG
369,6,94.965558,4.607,-74.130,Prioridad 1 - Intervención prioritaria,Hotspot persistente con severidad media-alta,60,152,2.533333,4,10,PUENTE ARANDA,TEJAR,AVENIDA DEL CONGRESO EUCARISTICO
27,7,94.701109,4.611,-74.075,Prioridad 1 - Intervención prioritaria,Hotspot severo,182,422,2.318681,4,4,SANTA FE,SANTA FE,AVENIDA CARACAS
147,8,94.636311,4.576,-74.120,Prioridad 1 - Intervención prioritaria,Hotspot persistente con severidad media-alta,92,236,2.565217,4,2,RAFAEL URIBE URIBE,QUIROGA SUR,AVENIDA CARACAS
515,9,94.575015,4.618,-74.170,Prioridad 1 - Intervención prioritaria,Hotspot persistente con severidad media-alta,48,132,2.750000,4,4,KENNEDY,CASABLANCA,AVENIDA AGOBERTO MEJIA CIFUENTES
343,10,94.455342,4.512,-74.115,Prioridad 1 - Intervención prioritaria,Hotspot persistente con severidad media-alta,68,160,2.352941,4,4,USME,LA ANDREA,AVENIDA CARACAS


In [38]:
resumen_prioridad_ipi = (
    zonas_avanzado
    .groupby("prioridad_IPI")
    .agg(
        cantidad_zonas=("prioridad_IPI", "count"),
        siniestros_acumulados=("cantidad_siniestros", "sum"),
        criticidad_total_acumulada=("criticidad_total", "sum"),
        muertes_registradas=("siniestros_con_muertos", "sum"),
        IPI_promedio=("IPI", "mean")
    )
    .reset_index()
    .sort_values("IPI_promedio", ascending=False)
)

resumen_prioridad_ipi

,prioridad_IPI,cantidad_zonas,siniestros_acumulados,criticidad_total_acumulada,muertes_registradas,IPI_promedio
0,Prioridad 1 - Intervención prioritaria,50,4426,10138,270,94.108920
1,Prioridad 2 - Auditoría de seguridad vial,150,9878,21086,490,91.891539
2,Prioridad 3 - Monitoreo y gestión preventiva,300,20439,38191,862,89.253322
3,Seguimiento,16630,226088,377664,2575,50.843946


In [39]:
top50_ipi = zonas_avanzado.head(50).copy()
top200_ipi = zonas_avanzado.head(200).copy()

zonas_avanzado.to_csv(
    ruta_outputs_reports / "zonas_criticas_diagnostico_avanzado_2016_2019.csv",
    index=False
)

top50_ipi.to_csv(
    ruta_outputs_reports / "top50_prioridad_intervencion_IPI_2016_2019.csv",
    index=False
)

top200_ipi.to_csv(
    ruta_outputs_reports / "top200_prioridad_intervencion_IPI_2016_2019.csv",
    index=False
)

resumen_prioridad_ipi.to_csv(
    ruta_outputs_reports / "resumen_prioridad_IPI_2016_2019.csv",
    index=False
)

print("Resultados avanzados guardados correctamente.")

Resultados avanzados guardados correctamente.


In [40]:
total_zonas = len(zonas_avanzado)
total_siniestros = zonas_avanzado["cantidad_siniestros"].sum()
total_criticidad = zonas_avanzado["criticidad_total"].sum()
total_muertes = zonas_avanzado["siniestros_con_muertos"].sum()

resumen_concentracion = []

for n in [50, 200, 500, 1000]:
    top_n = zonas_avanzado.head(n)
    
    resumen_concentracion.append({
        "top_n_zonas": n,
        "porcentaje_zonas": round((n / total_zonas) * 100, 2),
        "siniestros_acumulados": top_n["cantidad_siniestros"].sum(),
        "porcentaje_siniestros": round((top_n["cantidad_siniestros"].sum() / total_siniestros) * 100, 2),
        "criticidad_acumulada": top_n["criticidad_total"].sum(),
        "porcentaje_criticidad": round((top_n["criticidad_total"].sum() / total_criticidad) * 100, 2),
        "muertes_registradas": top_n["siniestros_con_muertos"].sum(),
        "porcentaje_muertes": round((top_n["siniestros_con_muertos"].sum() / total_muertes) * 100, 2)
    })

resumen_concentracion_ipi = pd.DataFrame(resumen_concentracion)

resumen_concentracion_ipi

,top_n_zonas,porcentaje_zonas,siniestros_acumulados,porcentaje_siniestros,criticidad_acumulada,porcentaje_criticidad,muertes_registradas,porcentaje_muertes
0,50,0.29,4426,1.70,10138,2.27,270,6.43
1,200,1.17,14304,5.48,31224,6.98,760,18.11
2,500,2.92,34743,13.32,69415,15.53,1622,38.65
3,1000,5.84,69604,26.69,124758,27.91,2826,67.33


In [41]:
resumen_concentracion_ipi.to_csv(
    ruta_outputs_reports / "resumen_concentracion_IPI_2016_2019.csv",
    index=False
)

print("Resumen de concentración IPI guardado correctamente.")

Resumen de concentración IPI guardado correctamente.


In [42]:
zonas_avanzado["rank_criticidad_total"] = zonas_avanzado["criticidad_total"].rank(
    ascending=False,
    method="first"
).astype(int)

zonas_avanzado["rank_volumen"] = zonas_avanzado["cantidad_siniestros"].rank(
    ascending=False,
    method="first"
).astype(int)

zonas_avanzado["rank_muertes"] = zonas_avanzado["siniestros_con_muertos"].rank(
    ascending=False,
    method="first"
).astype(int)

zonas_avanzado[[
    "rank_IPI",
    "rank_criticidad_total",
    "rank_volumen",
    "rank_muertes",
    "IPI",
    "cantidad_siniestros",
    "criticidad_total",
    "criticidad_promedio",
    "siniestros_con_muertos",
    "localidad_predominante",
    "via_predominante"
]].head(20)

,rank_IPI,rank_criticidad_total,rank_volumen,rank_muertes,IPI,cantidad_siniestros,criticidad_total,criticidad_promedio,siniestros_con_muertos,localidad_predominante,via_predominante
83,1,83,282,1,96.275540,108,296,2.740741,18,CIUDAD BOLIVAR,AVENIDA CIUDAD DE VILLAVICENCIO
135,2,135,347,6,95.778167,96,244,2.541667,10,SANTA FE,AVENIDA CARACAS
140,3,141,358,30,95.689434,94,238,2.531915,6,SANTA FE,AVENIDA FERNANDO MAZUERA
197,4,197,427,31,95.084647,86,206,2.395349,6,SANTA FE,AVENIDA FERNANDO MAZUERA
1224,5,1176,2562,86,94.994162,24,80,3.333333,4,BOSA,SIN_NMG
369,6,366,781,7,94.965558,60,152,2.533333,10,PUENTE ARANDA,AVENIDA DEL CONGRESO EUCARISTICO
27,7,28,90,87,94.701109,182,422,2.318681,4,SANTA FE,AVENIDA CARACAS
147,8,144,369,308,94.636311,92,236,2.565217,2,RAFAEL URIBE URIBE,AVENIDA CARACAS
515,9,503,1098,88,94.575015,48,132,2.750000,4,KENNEDY,AVENIDA AGOBERTO MEJIA CIFUENTES
343,10,341,643,89,94.455342,68,160,2.352941,4,USME,AVENIDA CARACAS


In [43]:
def clasificar_familia_analitica(row):
    rank_ipi = row["rank_IPI"]
    rank_criticidad = row["rank_criticidad_total"]
    rank_volumen = row["rank_volumen"]
    rank_muertes = row["rank_muertes"]
    
    if rank_ipi <= 200 and rank_criticidad <= 200:
        return "Hotspot robusto integral"
    
    elif rank_ipi <= 200 and rank_muertes <= 200 and rank_criticidad > 200:
        return "Hotspot de severidad/fatalidad"
    
    elif rank_criticidad <= 200 and rank_ipi > 200:
        return "Hotspot de carga acumulada"
    
    elif rank_ipi <= 500:
        return "Hotspot preventivo prioritario"
    
    else:
        return "Seguimiento"

zonas_avanzado["familia_analitica"] = zonas_avanzado.apply(
    clasificar_familia_analitica,
    axis=1
)

zonas_avanzado[[
    "rank_IPI",
    "rank_criticidad_total",
    "rank_volumen",
    "rank_muertes",
    "familia_analitica",
    "IPI",
    "cantidad_siniestros",
    "criticidad_total",
    "criticidad_promedio",
    "siniestros_con_muertos",
    "localidad_predominante",
    "barrio_predominante",
    "via_predominante"
]].head(50)

,rank_IPI,rank_criticidad_total,rank_volumen,rank_muertes,familia_analitica,IPI,cantidad_siniestros,criticidad_total,criticidad_promedio,siniestros_con_muertos,localidad_predominante,barrio_predominante,via_predominante
83,1,83,282,1,Hotspot robusto integral,96.275540,108,296,2.740741,18,CIUDAD BOLIVAR,VERONA,AVENIDA CIUDAD DE VILLAVICENCIO
135,2,135,347,6,Hotspot robusto integral,95.778167,96,244,2.541667,10,SANTA FE,VOTO NACIONAL,AVENIDA CARACAS
140,3,141,358,30,Hotspot robusto integral,95.689434,94,238,2.531915,6,SANTA FE,LA CAPUCHINA,AVENIDA FERNANDO MAZUERA
197,4,197,427,31,Hotspot robusto integral,95.084647,86,206,2.395349,6,SANTA FE,VERACRUZ,AVENIDA FERNANDO MAZUERA
1224,5,1176,2562,86,Hotspot de severidad/fatalidad,94.994162,24,80,3.333333,4,BOSA,EL CORZO I,SIN_NMG
369,6,366,781,7,Hotspot de severidad/fatalidad,94.965558,60,152,2.533333,10,PUENTE ARANDA,TEJAR,AVENIDA DEL CONGRESO EUCARISTICO
27,7,28,90,87,Hotspot robusto integral,94.701109,182,422,2.318681,4,SANTA FE,SANTA FE,AVENIDA CARACAS
147,8,144,369,308,Hotspot robusto integral,94.636311,92,236,2.565217,2,RAFAEL URIBE URIBE,QUIROGA SUR,AVENIDA CARACAS
515,9,503,1098,88,Hotspot de severidad/fatalidad,94.575015,48,132,2.750000,4,KENNEDY,CASABLANCA,AVENIDA AGOBERTO MEJIA CIFUENTES
343,10,341,643,89,Hotspot de severidad/fatalidad,94.455342,68,160,2.352941,4,USME,LA ANDREA,AVENIDA CARACAS


In [44]:
resumen_familia_analitica = (
    zonas_avanzado
    .groupby("familia_analitica")
    .agg(
        cantidad_zonas=("familia_analitica", "count"),
        siniestros_acumulados=("cantidad_siniestros", "sum"),
        criticidad_total_acumulada=("criticidad_total", "sum"),
        muertes_registradas=("siniestros_con_muertos", "sum"),
        IPI_promedio=("IPI", "mean")
    )
    .reset_index()
    .sort_values("IPI_promedio", ascending=False)
)

resumen_familia_analitica

,familia_analitica,cantidad_zonas,siniestros_acumulados,criticidad_total_acumulada,muertes_registradas,IPI_promedio
3,Hotspot robusto integral,45,6406,13106,268,92.879432
1,Hotspot de severidad/fatalidad,68,3408,7856,318,92.755039
2,Hotspot preventivo prioritario,346,16850,34862,873,89.934371
0,Hotspot de carga acumulada,155,32449,48741,371,83.958515
4,Seguimiento,16516,201718,342514,2367,50.628616


In [45]:
resumen_familia_analitica.to_csv(
    ruta_outputs_reports / "resumen_familia_analitica_2016_2019.csv",
    index=False
)

zonas_avanzado.to_csv(
    ruta_outputs_reports / "zonas_criticas_IPI_familia_analitica_2016_2019.csv",
    index=False
)

print("Familias analíticas guardadas correctamente.")

Familias analíticas guardadas correctamente.


In [46]:
zonas_sensibles_fatalidad = zonas_avanzado[
    (zonas_avanzado["siniestros_con_muertos"] >= 1) &
    (zonas_avanzado["cantidad_siniestros"] >= 20) &
    (zonas_avanzado["anios_activos"] >= 2)
].copy()

zonas_sensibles_fatalidad = zonas_sensibles_fatalidad.sort_values(
    "variacion_ranking_fatalidad",
    ascending=False
)

zonas_sensibles_fatalidad[[
    "LAT_ZONA",
    "LON_ZONA",
    "cantidad_siniestros",
    "criticidad_total",
    "criticidad_fatalidad_alta",
    "rank_base",
    "rank_fatalidad_alta",
    "variacion_ranking_fatalidad",
    "siniestros_con_muertos",
    "anios_activos",
    "localidad_predominante",
    "barrio_predominante",
    "via_predominante"
]].head(30)

,LAT_ZONA,LON_ZONA,cantidad_siniestros,criticidad_total,criticidad_fatalidad_alta,rank_base,rank_fatalidad_alta,variacion_ranking_fatalidad,siniestros_con_muertos,anios_activos,localidad_predominante,barrio_predominante,via_predominante
2765,4.722,-74.051,22,42,126,2687.0,1221.0,1466.0,4,4,SUBA,LOS CEDROS,AVENIDA PASEO DE LOS LIBERTADORES
4327,4.709,-74.080,20,28,68,4096.0,2783.0,1313.0,2,3,SUBA,NIZA SUR,AVENIDA BOYACA
4098,4.725,-74.124,20,28,68,4096.0,2783.0,1313.0,2,4,ENGATIVA,EL DORADO INDUSTRIAL,AVENIDA MEDELLIN
2294,4.564,-74.087,22,50,142,2218.0,1026.0,1192.0,4,4,SAN CRISTOBAL,MONTEBELLO,SIN_NMG
3662,4.660,-74.137,20,32,76,3611.0,2424.0,1187.0,2,4,FONTIBON,VEREDA EL TINTAL,AVENIDA CENTENARIO
3736,4.704,-74.134,20,32,76,3611.0,2424.0,1187.0,2,4,ENGATIVA,MARANDU,SIN_NMG
3756,4.718,-74.028,20,32,76,3611.0,2424.0,1187.0,2,4,USAQUEN,CEDRO NARVAEZ,AVENIDA ALBERTO LLERAS CAMARGO
3758,4.609,-74.128,20,32,76,3611.0,2424.0,1187.0,2,4,PUENTE ARANDA,TEJAR,AVENIDA PRIMERO DE MAYO
2202,4.682,-74.082,20,52,148,2116.0,951.0,1165.0,4,4,BARRIOS UNIDOS,METROPOLIS,AVENIDA DEL CONGRESO EUCARISTICO
4056,4.630,-74.130,22,30,70,3850.0,2685.0,1165.0,2,4,KENNEDY,MARSELLA,AVENIDA DE LAS AMERICAS


In [47]:
zonas_sensibles_fatalidad.to_csv(
    ruta_outputs_reports / "zonas_sensibles_fatalidad_2016_2019.csv",
    index=False
)

print("Tabla de zonas sensibles a fatalidad guardada correctamente.")

Tabla de zonas sensibles a fatalidad guardada correctamente.


## Hallazgo principal

El análisis permitió pasar de una visualización descriptiva de siniestros a un sistema de priorización espacial. Mediante el Índice de Prioridad de Intervención (IPI), se identificaron zonas que no necesariamente tienen el mayor volumen total de siniestros, pero sí presentan una combinación crítica de persistencia temporal, severidad promedio y presencia de siniestros fatales.

Este enfoque permitió distinguir tres tipos de zonas relevantes: hotspots robustos integrales, hotspots de severidad/fatalidad y hotspots de carga acumulada. Esta diferenciación es importante porque no todas las zonas críticas requieren el mismo tipo de intervención: algunas demandan gestión operativa por alto volumen, mientras que otras requieren auditorías de seguridad vial por severidad o fatalidad recurrente.

El resultado más accionable es que el universo de zonas agrupadas puede reducirse a listas operativas de intervención, como el Top 50, Top 200 y Top 500 del IPI, facilitando una priorización preliminar para toma de decisiones.

# Cierre del Notebook 02

## Resultado principal

En este notebook se construyó un sistema preliminar de priorización espacial de zonas críticas de siniestralidad vial en Bogotá para el periodo 2016–2019.

El análisis no se limita a identificar zonas con mayor cantidad de siniestros. También incorpora severidad, persistencia temporal y presencia de siniestros con muertos mediante el Índice de Prioridad de Intervención, IPI.

El objetivo del IPI es convertir una base amplia de zonas agrupadas en una lista operativa de intervención, útil para priorización exploratoria.

## Advertencia metodológica

Los resultados no deben interpretarse como una medición definitiva de riesgo vial, porque aún no se han incorporado variables de exposición como flujo vehicular, población, longitud de red vial, velocidades, infraestructura peatonal, semaforización o geometría vial.

Por lo tanto, se habla de criticidad y prioridad exploratoria, no de riesgo real definitivo.

In [48]:
# Crear versión más clara del resumen de concentración IPI

resumen_concentracion_ipi_final = resumen_concentracion_ipi.copy()

resumen_concentracion_ipi_final = resumen_concentracion_ipi_final.rename(columns={
    "muertes_registradas": "siniestros_con_muertos_acumulados",
    "porcentaje_muertes": "porcentaje_siniestros_con_muertos"
})

resumen_concentracion_ipi_final

,top_n_zonas,porcentaje_zonas,siniestros_acumulados,porcentaje_siniestros,criticidad_acumulada,porcentaje_criticidad,siniestros_con_muertos_acumulados,porcentaje_siniestros_con_muertos
0,50,0.29,4426,1.70,10138,2.27,270,6.43
1,200,1.17,14304,5.48,31224,6.98,760,18.11
2,500,2.92,34743,13.32,69415,15.53,1622,38.65
3,1000,5.84,69604,26.69,124758,27.91,2826,67.33


In [49]:
resumen_concentracion_ipi_final.to_csv(
    ruta_outputs_reports / "resumen_concentracion_IPI_2016_2019_final.csv",
    index=False
)

print("Resumen final de concentración IPI guardado.")

Resumen final de concentración IPI guardado.


In [50]:
columnas_top50_final = [
    "rank_IPI",
    "IPI",
    "LAT_ZONA",
    "LON_ZONA",
    "prioridad_IPI",
    "familia_analitica",
    "tipo_hotspot",
    "cantidad_siniestros",
    "criticidad_total",
    "criticidad_promedio",
    "anios_activos",
    "siniestros_solo_danos",
    "siniestros_con_heridos",
    "siniestros_con_muertos",
    "localidad_predominante",
    "barrio_predominante",
    "via_predominante",
    "clase_predominante",
    "gravedad_predominante"
]

top50_ipi_final = zonas_avanzado.head(50)[columnas_top50_final].copy()

top50_ipi_final.head(20)

,rank_IPI,IPI,LAT_ZONA,LON_ZONA,prioridad_IPI,familia_analitica,tipo_hotspot,cantidad_siniestros,criticidad_total,criticidad_promedio,anios_activos,siniestros_solo_danos,siniestros_con_heridos,siniestros_con_muertos,localidad_predominante,barrio_predominante,via_predominante,clase_predominante,gravedad_predominante
83,1,96.275540,4.576,-74.155,Prioridad 1 - Intervención prioritaria,Hotspot robusto integral,Hotspot persistente con severidad media-alta,108,296,2.740741,4,32,58,18,CIUDAD BOLIVAR,VERONA,AVENIDA CIUDAD DE VILLAVICENCIO,CHOQUE,CON HERIDOS
135,2,95.778167,4.602,-74.081,Prioridad 1 - Intervención prioritaria,Hotspot robusto integral,Hotspot persistente con severidad media-alta,96,244,2.541667,4,32,54,10,SANTA FE,VOTO NACIONAL,AVENIDA CARACAS,ATROPELLO,CON HERIDOS
140,3,95.689434,4.602,-74.077,Prioridad 1 - Intervención prioritaria,Hotspot robusto integral,Hotspot persistente con severidad media-alta,94,238,2.531915,4,28,60,6,SANTA FE,LA CAPUCHINA,AVENIDA FERNANDO MAZUERA,ATROPELLO,CON HERIDOS
197,4,95.084647,4.605,-74.075,Prioridad 1 - Intervención prioritaria,Hotspot robusto integral,Hotspot persistente con severidad media-alta,86,206,2.395349,4,32,48,6,SANTA FE,VERACRUZ,AVENIDA FERNANDO MAZUERA,CHOQUE,CON HERIDOS
1224,5,94.994162,4.642,-74.191,Prioridad 1 - Intervención prioritaria,Hotspot de severidad/fatalidad,Hotspot persistente con severidad media-alta,24,80,3.333333,4,0,20,4,BOSA,EL CORZO I,SIN_NMG,CHOQUE,CON HERIDOS
369,6,94.965558,4.607,-74.130,Prioridad 1 - Intervención prioritaria,Hotspot de severidad/fatalidad,Hotspot persistente con severidad media-alta,60,152,2.533333,4,24,26,10,PUENTE ARANDA,TEJAR,AVENIDA DEL CONGRESO EUCARISTICO,CHOQUE,CON HERIDOS
27,7,94.701109,4.611,-74.075,Prioridad 1 - Intervención prioritaria,Hotspot robusto integral,Hotspot severo,182,422,2.318681,4,66,112,4,SANTA FE,SANTA FE,AVENIDA CARACAS,CHOQUE,CON HERIDOS
147,8,94.636311,4.576,-74.120,Prioridad 1 - Intervención prioritaria,Hotspot robusto integral,Hotspot persistente con severidad media-alta,92,236,2.565217,4,22,68,2,RAFAEL URIBE URIBE,QUIROGA SUR,AVENIDA CARACAS,ATROPELLO,CON HERIDOS
515,9,94.575015,4.618,-74.170,Prioridad 1 - Intervención prioritaria,Hotspot de severidad/fatalidad,Hotspot persistente con severidad media-alta,48,132,2.750000,4,10,34,4,KENNEDY,CASABLANCA,AVENIDA AGOBERTO MEJIA CIFUENTES,CHOQUE,CON HERIDOS
343,10,94.455342,4.512,-74.115,Prioridad 1 - Intervención prioritaria,Hotspot de severidad/fatalidad,Hotspot persistente con severidad media-alta,68,160,2.352941,4,26,38,4,USME,LA ANDREA,AVENIDA CARACAS,CHOQUE,CON HERIDOS


In [51]:
top50_ipi_final.to_csv(
    ruta_outputs_reports / "top50_IPI_final_2016_2019.csv",
    index=False
)

print("Top 50 IPI final guardado correctamente.")

Top 50 IPI final guardado correctamente.


In [52]:
top50_mapa_ipi = top50_ipi_final.copy()

mapa_ipi_final = folium.Map(
    location=[4.65, -74.08],
    zoom_start=11,
    tiles="CartoDB positron"
)

max_ipi = top50_mapa_ipi["IPI"].max()

for _, row in top50_mapa_ipi.iterrows():
    
    radio = 6 + (row["IPI"] / max_ipi) * 18
    
    popup_texto = f"""
    <b>Zona priorizada por IPI</b><br><br>
    <b>Rank IPI:</b> {row['rank_IPI']}<br>
    <b>IPI:</b> {round(row['IPI'], 2)}<br>
    <b>Prioridad:</b> {row['prioridad_IPI']}<br>
    <b>Familia analítica:</b> {row['familia_analitica']}<br>
    <b>Tipo de hotspot:</b> {row['tipo_hotspot']}<br><br>
    
    <b>Localidad:</b> {row['localidad_predominante']}<br>
    <b>Barrio:</b> {row['barrio_predominante']}<br>
    <b>Vía predominante:</b> {row['via_predominante']}<br><br>
    
    <b>Cantidad de siniestros:</b> {row['cantidad_siniestros']}<br>
    <b>Criticidad total:</b> {row['criticidad_total']}<br>
    <b>Criticidad promedio:</b> {round(row['criticidad_promedio'], 2)}<br>
    <b>Años activos:</b> {row['anios_activos']}<br><br>
    
    <b>Solo daños:</b> {row['siniestros_solo_danos']}<br>
    <b>Con heridos:</b> {row['siniestros_con_heridos']}<br>
    <b>Con muertos:</b> {row['siniestros_con_muertos']}<br><br>
    
    <b>Clase predominante:</b> {row['clase_predominante']}<br>
    <b>Gravedad predominante:</b> {row['gravedad_predominante']}<br><br>
    <b>Coordenada aproximada:</b> {row['LAT_ZONA']}, {row['LON_ZONA']}
    """
    
    folium.CircleMarker(
        location=[row["LAT_ZONA"], row["LON_ZONA"]],
        radius=radio,
        popup=folium.Popup(popup_texto, max_width=400),
        fill=True,
        fill_opacity=0.7
    ).add_to(mapa_ipi_final)

mapa_ipi_final

In [53]:
mapa_ipi_final.save(
    ruta_outputs_maps / "mapa_top50_IPI_final_2016_2019.html"
)

print("Mapa final Top 50 IPI guardado correctamente.")

Mapa final Top 50 IPI guardado correctamente.


In [54]:
zonas_avanzado.to_csv(
    ruta_outputs_reports / "zonas_criticas_IPI_completo_2016_2019.csv",
    index=False
)

resumen_familia_analitica.to_csv(
    ruta_outputs_reports / "resumen_familia_analitica_2016_2019.csv",
    index=False
)

resumen_prioridad_ipi.to_csv(
    ruta_outputs_reports / "resumen_prioridad_IPI_2016_2019.csv",
    index=False
)

zonas_sensibles_fatalidad.to_csv(
    ruta_outputs_reports / "zonas_sensibles_fatalidad_2016_2019.csv",
    index=False
)

print("Tablas finales del Notebook 02 guardadas correctamente.")

Tablas finales del Notebook 02 guardadas correctamente.


In [55]:
resumen_md = f"""
# Resumen Ejecutivo - Notebook 02

## Proyecto
VíaSegura AI

## Notebook
02_indice_criticidad_y_hotspots.ipynb

## Periodo analizado
2016–2019

## Base utilizada
data/processed/accidentes_bogota_2016_2019_limpio.csv

## Registros analizados
{len(df):,} siniestros viales georreferenciados.

## Objetivo del notebook
Construir un sistema preliminar de priorización espacial de zonas críticas de siniestralidad vial en Bogotá mediante el Índice de Prioridad de Intervención, IPI.

## Variables consideradas en el IPI
- Volumen de siniestros.
- Criticidad total.
- Severidad promedio.
- Persistencia temporal.
- Presencia de siniestros con muertos.

## Resultado principal
El análisis permitió pasar de una visualización descriptiva de siniestros a un sistema de priorización espacial. El IPI permite identificar zonas que no necesariamente tienen el mayor volumen de siniestros, pero sí presentan una combinación crítica de persistencia, severidad y presencia de siniestros con muertos.

## Concentración del Top 50 IPI
- Zonas priorizadas: 50.
- Porcentaje de zonas: {resumen_concentracion_ipi_final.loc[resumen_concentracion_ipi_final['top_n_zonas'] == 50, 'porcentaje_zonas'].values[0]}%.
- Siniestros acumulados: {resumen_concentracion_ipi_final.loc[resumen_concentracion_ipi_final['top_n_zonas'] == 50, 'siniestros_acumulados'].values[0]:,}.
- Criticidad acumulada: {resumen_concentracion_ipi_final.loc[resumen_concentracion_ipi_final['top_n_zonas'] == 50, 'criticidad_acumulada'].values[0]:,}.
- Siniestros con muertos acumulados: {resumen_concentracion_ipi_final.loc[resumen_concentracion_ipi_final['top_n_zonas'] == 50, 'siniestros_con_muertos_acumulados'].values[0]:,}.

## Familias analíticas
Se clasificaron las zonas en:
- Hotspot robusto integral.
- Hotspot de severidad/fatalidad.
- Hotspot de carga acumulada.
- Hotspot preventivo prioritario.
- Seguimiento.

## Limitaciones
Los resultados no representan una medición definitiva de riesgo vial. Para estimar riesgo real se requiere incorporar exposición, población, flujos vehiculares, longitud de red vial, velocidad, geometría, infraestructura peatonal, semaforización y condiciones urbanas.

## Siguiente paso recomendado
Construir el Notebook 03 para validar relevancia actual mediante datos recientes, idealmente 2022–2024 o 2023–2025, y comparar la persistencia de los hotspots identificados en el periodo base 2016–2019.
"""

ruta_resumen_md = ruta_outputs_reports / "resumen_ejecutivo_notebook_02.md"

with open(ruta_resumen_md, "w", encoding="utf-8") as f:
    f.write(resumen_md)

print("Resumen ejecutivo del Notebook 02 guardado en:")
print(ruta_resumen_md)

Resumen ejecutivo del Notebook 02 guardado en:
C:\Users\jorge\Documents\viasegura_ai\outputs\reports\resumen_ejecutivo_notebook_02.md


In [56]:
archivos_esperados_notebook_02 = [
    ruta_outputs_reports / "zonas_criticas_siniestros_2016_2019.csv",
    ruta_outputs_reports / "top20_zonas_criticas_siniestros_2016_2019.csv",
    ruta_outputs_reports / "zonas_criticas_diagnostico_2016_2019.csv",
    ruta_outputs_reports / "resumen_tipo_hotspot_2016_2019.csv",
    ruta_outputs_reports / "resumen_prioridad_intervencion_2016_2019.csv",
    ruta_outputs_reports / "zonas_criticas_diagnostico_avanzado_2016_2019.csv",
    ruta_outputs_reports / "top50_prioridad_intervencion_IPI_2016_2019.csv",
    ruta_outputs_reports / "top200_prioridad_intervencion_IPI_2016_2019.csv",
    ruta_outputs_reports / "resumen_prioridad_IPI_2016_2019.csv",
    ruta_outputs_reports / "resumen_concentracion_IPI_2016_2019_final.csv",
    ruta_outputs_reports / "resumen_familia_analitica_2016_2019.csv",
    ruta_outputs_reports / "zonas_sensibles_fatalidad_2016_2019.csv",
    ruta_outputs_reports / "top50_IPI_final_2016_2019.csv",
    ruta_outputs_reports / "zonas_criticas_IPI_completo_2016_2019.csv",
    ruta_outputs_reports / "resumen_ejecutivo_notebook_02.md",
    ruta_outputs_maps / "mapa_top50_zonas_criticas_siniestros_2016_2019.html",
    ruta_outputs_maps / "mapa_top50_IPI_final_2016_2019.html"
]

manifiesto_outputs = []

for archivo in archivos_esperados_notebook_02:
    manifiesto_outputs.append({
        "archivo": str(archivo),
        "existe": archivo.exists()
    })

manifiesto_outputs = pd.DataFrame(manifiesto_outputs)

manifiesto_outputs

,archivo,existe
0,C:\Users\jorge\Documents\viasegura_ai\outputs\...,True
1,C:\Users\jorge\Documents\viasegura_ai\outputs\...,True
2,C:\Users\jorge\Documents\viasegura_ai\outputs\...,True
3,C:\Users\jorge\Documents\viasegura_ai\outputs\...,True
4,C:\Users\jorge\Documents\viasegura_ai\outputs\...,True
5,C:\Users\jorge\Documents\viasegura_ai\outputs\...,True
6,C:\Users\jorge\Documents\viasegura_ai\outputs\...,True
7,C:\Users\jorge\Documents\viasegura_ai\outputs\...,True
8,C:\Users\jorge\Documents\viasegura_ai\outputs\...,True
9,C:\Users\jorge\Documents\viasegura_ai\outputs\...,True


In [57]:
manifiesto_outputs.to_csv(
    ruta_outputs_reports / "manifiesto_outputs_notebook_02.csv",
    index=False
)

print("Manifiesto de outputs guardado correctamente.")

Manifiesto de outputs guardado correctamente.


# Conclusión del Notebook 02

El Notebook 02 permitió transformar la base limpia de siniestros viales 2016–2019 en un sistema preliminar de priorización espacial.

El principal avance fue la construcción del Índice de Prioridad de Intervención, IPI. Este índice combina volumen de siniestros, criticidad total, severidad promedio, persistencia temporal y presencia de siniestros con muertos.

A diferencia de un ranking simple por cantidad de siniestros, el IPI permite identificar zonas que pueden tener menor volumen, pero mayor severidad relativa o mayor presencia de fatalidad. Esto hace que la priorización sea más útil para orientar auditorías de seguridad vial, revisiones de infraestructura o análisis detallados de intervención.

El resultado más importante es la generación de listas operativas como el Top 50, Top 200 y Top 500 de zonas priorizadas. Estas listas permiten pasar de una lectura descriptiva del problema a una posible herramienta de toma de decisiones.

Sin embargo, el análisis sigue siendo exploratorio. Para convertirlo en una medición más cercana al riesgo vial real, será necesario incorporar variables de exposición, como población, flujos vehiculares, longitud de red vial, velocidades, geometría, infraestructura peatonal y condiciones de operación.

El siguiente paso del proyecto será construir un Notebook 03 enfocado en la validación temporal y actualidad del método, comparando el periodo base 2016–2019 con datos recientes.